# Optimización y Guardado del Modelo — Optuna
## ML Modeling Challenge - Wizeline

Optimización bayesiana de hiperparámetros con **Optuna** para el modelo ganador (**CatBoost**),
utilizando las features con importancia > 5 obtenidas en la sección 7 del Notebook 2.

**Ventajas de Optuna sobre RandomizedSearchCV:**
- **Búsqueda bayesiana** (TPE sampler): aprende qué zonas del espacio son prometedoras y las prioriza
- **Pruning**: cancela trials que claramente no mejorarán → más eficiente
- Visualizaciones interactivas del proceso de optimización

**Métrica de selección**: menor **SMAPE de CV**

## 0. Imports y configuración

In [1]:
from pathlib import Path

import joblib
import numpy as np
import optuna
import plotly.express as px
import polars as pl
import yaml
from catboost import CatBoostRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold, cross_validate

from src.pipeline.modeling.train_functions import (
    SEED,
    build_pipeline,
    fugacity_smape_score,
    mape_score,
    run_cross_validation,
    smape_score,
)

# Silenciar logs de Optuna — mostrar solo trials completados
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 20  # Aumentar para búsqueda más exhaustiva (ej. 100)

print(f'Seed global:    {SEED}')
print(f'Trials Optuna:  {N_TRIALS}')
print(f'Optuna version: {optuna.__version__}')


/home/miguel_uicab/DOCUMENTS/ML_Modeling_Challenge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Seed global:    5000
Trials Optuna:  20
Optuna version: 4.8.0


## 1. Carga de datos y selección de features

Features leídas desde `config.yaml` (importancia > 5 del modelo ganador CatBoost v2).

In [2]:
with open('config.yaml') as f:
    config = yaml.safe_load(f)

FEATURES: list[str] = config['features']['selected']

df_train = pl.read_csv(config['data']['train_path'])
X = df_train.select(FEATURES).to_numpy()
y = df_train['target'].to_numpy()

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')


Features (5): ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
X shape: (800, 5)
y shape: (800,)


## 2. Función objetivo

La función objetivo recibe un `trial` de Optuna, propone hiperparámetros con `trial.suggest_*`
y devuelve el **SMAPE CV promedio** (a minimizar).

| Método | Tipo | Descripción |
|---|---|---|
| `trial.suggest_int` | entero | `iterations`, `depth` |
| `trial.suggest_float(..., log=True)` | float log-uniforme | `learning_rate` |
| `trial.suggest_float` | float uniforme | `l2_leaf_reg`, `subsample` |

In [3]:
kf = KFold(n_splits=4, shuffle=True, random_state=SEED)
smape_scorer = make_scorer(smape_score, greater_is_better=False)
mape_scorer = make_scorer(mape_score, greater_is_better=False)

scoring = {
    'smape': smape_scorer,
    'mape': mape_scorer,
}


def objective(trial: optuna.Trial) -> float:
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_seed': SEED,
        'verbose': 0,
    }

    pipeline = build_pipeline(CatBoostRegressor(**params))
    cv_raw = cross_validate(pipeline, X, y, cv=kf, scoring=scoring)
    smape_cv = float(-np.mean(cv_raw['test_smape']))
    return smape_cv


print('Función objetivo definida.')
print(f'Métrica a minimizar: SMAPE CV  |  Folds: {kf.n_splits}  |  Trials: {N_TRIALS}')


Función objetivo definida.
Métrica a minimizar: SMAPE CV  |  Folds: 4  |  Trials: 20


## 3. Optimización con Optuna

**TPESampler** (Tree-structured Parzen Estimator): el sampler bayesiano por defecto de Optuna.
Aprende a partir de los trials anteriores cuáles regiones del espacio dan mejores resultados.

In [4]:
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction='minimize', sampler=sampler, study_name='catboost-smape')

print(f'Iniciando optimización: {N_TRIALS} trials...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\nOptimización completada.')
print(f'Mejor SMAPE CV : {study.best_value:.4f}%')
print(f'Mejor trial    : #{study.best_trial.number}')


Iniciando optimización: 20 trials...


Best trial: 16. Best value: 11.268: 100%|██████████| 20/20 [01:34<00:00,  4.74s/it]


Optimización completada.
Mejor SMAPE CV : 11.2680%
Mejor trial    : #16


## 4. Análisis de resultados

In [5]:
best_params: dict = study.best_trial.params

print(f'Mejor SMAPE CV : {study.best_value:.4f}%')
print(f'Mejor trial    : #{study.best_trial.number}')
print('\nMejores hiperparámetros:')
for k, v in best_params.items():
    print(f'  {k}: {v}')


Mejor SMAPE CV : 11.2680%
Mejor trial    : #16

Mejores hiperparámetros:
  iterations: 867
  learning_rate: 0.05268969892790417
  depth: 5
  l2_leaf_reg: 9.922850737939058
  subsample: 0.9338494784637711


In [6]:
# Historial de todos los trials
trials_df = pl.DataFrame(
    {
        'trial': [t.number for t in study.trials],
        'smape_cv': [t.value for t in study.trials],
        **{k: [t.params[k] for t in study.trials] for k in best_params},
    }
).sort('smape_cv')

print('Top 10 trials por SMAPE CV:')
print(trials_df.head(10))


Top 10 trials por SMAPE CV:
shape: (10, 7)
┌───────┬───────────┬────────────┬───────────────┬───────┬─────────────┬───────────┐
│ trial ┆ smape_cv  ┆ iterations ┆ learning_rate ┆ depth ┆ l2_leaf_reg ┆ subsample │
│ ---   ┆ ---       ┆ ---        ┆ ---           ┆ ---   ┆ ---         ┆ ---       │
│ i64   ┆ f64       ┆ i64        ┆ f64           ┆ i64   ┆ f64         ┆ f64       │
╞═══════╪═══════════╪════════════╪═══════════════╪═══════╪═════════════╪═══════════╡
│ 16    ┆ 11.267968 ┆ 867        ┆ 0.05269       ┆ 5     ┆ 9.922851    ┆ 0.933849  │
│ 18    ┆ 11.272386 ┆ 846        ┆ 0.046912      ┆ 4     ┆ 8.391181    ┆ 0.90923   │
│ 9     ┆ 11.288627 ┆ 972        ┆ 0.026574      ┆ 6     ┆ 6.566835    ┆ 0.912695  │
│ 17    ┆ 11.374093 ┆ 1486       ┆ 0.02646       ┆ 7     ┆ 9.768514    ┆ 0.949137  │
│ 12    ┆ 11.393821 ┆ 699        ┆ 0.064277      ┆ 5     ┆ 3.628621    ┆ 0.749924  │
│ 15    ┆ 11.42326  ┆ 648        ┆ 0.023019      ┆ 7     ┆ 4.700226    ┆ 0.692646  │
│ 13    ┆ 11.517428 ┆ 

In [7]:
# Curva de convergencia: mejor SMAPE acumulado por trial
best_so_far = []
current_best = float('inf')
for t in study.trials:
    if t.value < current_best:
        current_best = t.value
    best_so_far.append(current_best)

fig = px.line(
    x=list(range(len(best_so_far))),
    y=best_so_far,
    title='Convergencia de Optuna — Mejor SMAPE CV acumulado',
    labels={'x': 'Trial #', 'y': 'Mejor SMAPE CV (%)'},
    markers=True,
)
fig.update_layout(height=400)
fig.show()


In [8]:
# Importancia de hiperparámetros según Optuna
importance = optuna.importance.get_param_importances(study)
importance_df = pl.DataFrame(
    {'hyperparameter': list(importance.keys()), 'importance': list(importance.values())}
).sort('importance', descending=True)

fig = px.bar(
    importance_df,
    x='importance',
    y='hyperparameter',
    orientation='h',
    text_auto='.4f',
    color='importance',
    color_continuous_scale='Viridis',
    title='Importancia de hiperparámetros — Optuna FAnova',
    labels={'importance': 'Importancia relativa', 'hyperparameter': 'Hiperparámetro'},
)
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=400,
)
fig.show()


## 5. Ajuste final y guardado del modelo

Con los mejores hiperparámetros encontrados se entrena el pipeline sobre el dataset completo y se persiste en disco.


In [9]:
best_params = study.best_trial.params

# Agrega parámetros fijos que no se optimizan
catboost_params = {**best_params, 'random_seed': SEED, 'verbose': 0}

best_pipeline = build_pipeline(CatBoostRegressor(**catboost_params))
best_pipeline.fit(X, y)

model_path = Path(config['model']['artifact_path'])
model_path.parent.mkdir(exist_ok=True)
joblib.dump(best_pipeline, model_path)

print(f'Modelo guardado en : {model_path}')
print(f'Tamaño             : {model_path.stat().st_size / 1024:.1f} KB')


Modelo guardado en : models/catboost_optimized.joblib
Tamaño             : 508.8 KB


## 6. Evaluación del modelo optimizado

Validación cruzada final con todas las métricas para confirmar la calidad del modelo.


In [10]:
eval_pipeline = build_pipeline(CatBoostRegressor(**catboost_params))
metrics = run_cross_validation(eval_pipeline, X, y, n_splits=5, seed=SEED)

rmse_cv = metrics['rmse_cv']
mae_cv = metrics['mae_cv']
mape_cv = metrics['mape_cv']
smape_cv = metrics['smape_cv']
r2_cv = metrics['r2_cv']
fugacity_smape_cv = metrics['fugacity_smape_cv']

print('=' * 50)
print('  Métricas de validación cruzada (5-fold CV)')
print('=' * 50)
print(f'  RMSE              : {rmse_cv:.4f}')
print(f'  MAE               : {mae_cv:.4f}')
print(f'  MAPE (%)          : {mape_cv:.2f}%')
print(f'  SMAPE (%)         : {smape_cv:.2f}%')
print(f'  R²                : {r2_cv:.4f}')
print(f'  Fugacity SMAPE (%) : {fugacity_smape_cv:.2f}%')
print('=' * 50)


  Métricas de validación cruzada (5-fold CV)
  RMSE              : 1.7113
  MAE               : 1.3762
  MAPE (%)          : 14.17%
  SMAPE (%)         : 11.29%
  R²                : 0.8840
  Fugacity SMAPE (%) : 22.88%


In [11]:
df_metrics = pl.DataFrame(
    {
        'Métrica': ['RMSE', 'MAE', 'MAPE (%)', 'SMAPE (%)', 'R²', 'Fugacity SMAPE (%)'],
        'Valor': [rmse_cv, mae_cv, mape_cv, smape_cv, r2_cv, fugacity_smape_cv],
    }
)

fig = px.bar(
    df_metrics,
    x='Métrica',
    y='Valor',
    text_auto='.4f',
    color='Métrica',
    title='Métricas de validación cruzada — Modelo optimizado con Optuna',
    labels={'Valor': 'Valor promedio (5-fold CV)'},
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=450)
fig.show()
